## Submissions for movies dataset

In [1]:
from preprocessing_class import Preprocessing, load_movies,load_movies_test
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression,LogisticRegressionCV
from sklearn.svm import LinearSVC,SVC
from nltk.corpus import stopwords
import numpy as np
import string
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

#### Format submission
```
submission-movie-X.csv
```

In [19]:
# load dataset train
path = "./Dataset/movies1000/"
alltxt,alllabs = load_movies(path)
alltxt, alllabs = np.array(alltxt), np.array(alllabs)

In [20]:
# load dataset test
alltxt_test = load_movies_test("Test_set/testSentiment.txt")
alltxt_test = np.array(alltxt_test)

In [ ]:
alltxt_test.shape, alltxt.shape

((25000,), (2000,))

In [22]:
alltxt_test[0],alltxt[0]

(np.str_("Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.\n"),
 np.str_(' " holy man " boasts a sweet , gentle , comic performance from an unusually subdued eddie murphy and a few moderately funny skits . \nunfortunately , to get to the good stuff you have to sit through a painfully long set-up , loads of tedious filler , interminable shots of jeff goldblum stammering and twitching , a sup

### Submission 1

In [23]:
def build_vectorizer(name="count", stop_words=None, max_features=1000,
                     lowercase=False, strip_accents=None, ngram_range=(1,1), use_idf= True, smooth_idf=True, sublinear_tf=False, 
                     max_df=0.95, min_df=2, preprocessor = lambda x: x):
    if name == "count":
        vectorizer = CountVectorizer(stop_words=stop_words,max_features=max_features,max_df=max_df,min_df=min_df,ngram_range=ngram_range,lowercase=lowercase,
                                     strip_accents=strip_accents, preprocessor=preprocessor)    
        
    elif name == "tfidf":
        vectorizer = TfidfVectorizer(use_idf= use_idf, smooth_idf=smooth_idf, sublinear_tf=sublinear_tf,max_df=max_df, min_df=min_df, max_features=max_features,
                                     stop_words=stop_words, ngram_range=ngram_range,lowercase=lowercase,strip_accents=strip_accents, preprocessor=preprocessor)
    
    return vectorizer
    ## to implement more embeddings 

def build_embeddings():
    pass

def vectorize_txts(txts,vectorizer):
    X = vectorizer.fit_transform(txts)
    return X

def build_model(name="kmeans", n_clusters = 2, max_iter = 1000, penalty='l2', loss='squared_hinge',kernel='rbf',solver='lbfgs'):
    if name == "kmeans":
        return KMeans(n_clusters=n_clusters,max_iter=max_iter) 
    elif name == "linear_svm":
        return LinearSVC(penalty=penalty,loss=loss,max_iter=max_iter)
    elif name == "svm":
        return SVC(max_iter=max_iter,kernel=kernel)
    elif name == "logreg":
        return LogisticRegression(solver=solver,max_iter=max_iter) 

In [24]:
punc = set(string.punctuation + '\n\r\t')
custom_punctuation = "".join(punc)

prep = Preprocessing(low_case = True,
                rm_punctuation = True,
                rm_number = False,
                word_norm = None, # stem faster
                pos_tagging = False, # to check
                all_capital = True, # keep all capital words as they are
                cap_name = True, # garder les noms en majuscules, for pos tag
                rm_accent = False, 
                lang = "english",
                punct = custom_punctuation, # punctuation can contain -, that would be kept
                urls = False 
                )

In [ ]:
final_stopwords_list = stopwords.words('english') 
vectoriser = build_vectorizer(name="tfidf", stop_words=final_stopwords_list, max_features=None, preprocessor=prep)

# train on the sub train
vect_train = vectoriser.fit_transform(alltxt)

vect_test = vectoriser.transform(alltxt_test)

In [26]:
# model logistic regression
model = build_model(name="logreg")

In [57]:
# train on training set
model.fit(vect_train,alllabs)

# preds
y_pred_train = model.predict(vect_train)
y_pred_train_prob = model.predict_proba(vect_train) # probas for ROC

y_pred_test = model.predict(vect_test)
y_pred_test_prob = model.predict_proba(vect_test)

In [79]:
n=1000
y_pred_train[n],alllabs[n],y_pred_train_prob[n,:],alltxt[n]

(np.int64(1),
 np.int64(1),
 array([0.24153261, 0.75846739]),
 np.str_('tibet has entered the american consciousness slowly during the past few years and burst into the forefront recently . \nthe dalai lama has made speaking tours , tibetan designs and artifacts are showing up in shops , the plight of the people has captured the imagination of major actors and artists and now there are two big budget films within as many months . \n " kundun " ( " presence " , a name the monks call the young dalai lama ) opens in 1937 with the search for the 14th incarnation of the tibetan leader . \na group of monks enters a village and a two-year-old child catches their attention . \nthe young boy proves himself by identifying possessions of the previous holy man as " mine " . \nhe and his family are then moved to a monastery in lhassa where he is to be prepared to take his position as the religious and political leader of the country . \nhe is forced into difficult decisions when china invades the c

In [84]:
# dim 0 = 0 = N, dim 1 = 1 = P

n = 20
y_pred_test[n],y_pred_test_prob[n,:],alltxt_test[n]


(np.int64(1),
 array([0.32629319, 0.67370681]),
 np.str_('You know, Robin Williams, God bless him, is constantly shooting himself in the foot lately with all these dumb comedies he has done this decade (with perhaps the exception of "Death To Smoochy", which bombed when it came out but is now a cult classic). The dramas he has made lately have been fantastic, especially "Insomnia" and "One Hour Photo". "The Night Listener", despite mediocre reviews and a quick DVD release, is among his best work, period.<br /><br />This is a very chilling story, even though it doesn\'t include a serial killer or anyone that physically dangerous for that matter. The concept of the film is based on an actual case of fraud that still has yet to be officially confirmed. In high school, I read an autobiography by a child named Anthony Godby Johnson, who suffered horrific abuse and eventually contracted AIDS as a result. I was moved by the story until I read reports online that Johnson may not actually exist

In [ ]:
from sklearn.metrics import f1_score

def save_submission(predictions, number):
    filename = f"Test_submission/movie/submission-movie-{number}.csv"
    np.savetxt(filename, predictions, delimiter="\n", fmt='%s')

def model_prediction(model, vectorizer, X_train, y_train, X_test, number = 'test'):
    # vectorizing
    vect_train = vectorizer.fit_transform(X_train)
    vect_test = vectorizer.transform(X_test)

    # scaling ?

    model.fit(vect_train, y_train)

    # predictions
    y_pred_train = model.predict(vect_train)

    y_pred = model.predict(vect_test)
    final_pred = np.where(y_pred == 0, 'N', 'P')
    print(final_pred[:5])
    
    # prints
    val, count = np.unique(final_pred, return_counts=True) # seems like unbalanced, or biaised
    print("F1 score of train",f1_score(y_train,y_pred_train))
    print("Distribution of test", val, count)

    # saving
    save_submission(final_pred, number)

In [94]:
punc = set(string.punctuation + '\n\r\t')
custom_punctuation = "".join(punc)

prep = Preprocessing(low_case = True,
                rm_punctuation = True,
                rm_number = False,
                word_norm = None, # stem faster
                pos_tagging = False, # to check
                all_capital = True, # keep all capital words as they are
                cap_name = True, # garder les noms en majuscules, for pos tag
                rm_accent = False, 
                lang = "english",
                punct = custom_punctuation, # punctuation can contain -, that would be kept
                urls = False 
                )

In [95]:
model = build_model(name="logreg")

final_stopwords_list = stopwords.words('english') 
vectoriser = build_vectorizer(name="tfidf", stop_words=final_stopwords_list, max_features=None, preprocessor=prep)

In [96]:
model_prediction(model, vectoriser, alltxt, alllabs, alltxt_test, number=1)

['P' 'P' 'N' 'N' 'P']
F1 score of train 0.975
Distribution of test ['N' 'P'] [13314 11686]


### Submission 2